## Ch8-02 — Violation witness

This notebook introduces the violation witness pattern using `verify_satisfaction()`; after running it you can show that the slow variant's 200-second cycle time violates the TimelyToast requirement and record the failing verdict as a ReviewRecord.


Notebook 01 showed that `slow` holds=False for the TimelyToast requirement. This notebook uses the failing Verdict as an explicit violation witness: it extracts the failing verdict, creates a ReviewRecord that references it, and validates the record. A violation witness is engineering evidence — it establishes that the requirement boundary is real and that the slow design falls outside it. See [Ch8-01 satisfaction evaluation](01-invariant-def.ipynb) for the `verify_satisfaction()` call.


In [ ]:
import opensysml
from toaster.report import format_diagnostics

source = """package ToasterDemo {
    private import ScalarValues::*;

    abstract part def ToastingSystem {
        doc /* Transform bread into toast acceptable to its user. */
    }
    part def Heater { attribute power : Real default = 800.0; }
    part def HeatingSystem :> ToastingSystem;
    part def ControlSystem :> ToastingSystem;
    part def Toaster {
        attribute cycleTime : Real default = 120.0;
        part heating : HeatingSystem;
        part control : ControlSystem;
    }
    part nominal : Toaster;
    part slow : Toaster { attribute :>> cycleTime = 200.0; }
    requirement def TimelyToast {
        subject toaster : Toaster;
        require constraint { toaster.cycleTime <= 180.0 }
    }
    requirement timely : TimelyToast;
    part evidence {
        assert satisfy timely by nominal;
        assert satisfy timely by slow;
    }
    calc def DeliveredEnergy {
        in power : Real;
        in duration : Real;
        in efficiency : Real;
        return : Real = power * duration * efficiency;
    }
    action def ApplyHeat {
        in power : Real;
        in duration : Real;
        in efficiency : Real;
        out energy : Real;
        first start;
        then action calculate {
            assign energy := DeliveredEnergy(power, duration, efficiency);
        }
        then done;
    }
    item def Start;
    item def Finish;
    item def Cancel;
    allocate ApplyHeat to HeatingSystem;
    requirement def HeatingReq {
        subject heater : Heater;
        require constraint { heater.power >= 600.0 }
    }
    requirement heating : HeatingReq;
    part efficient : Heater;
    part weak : Heater { attribute :>> power = 400.0; }
    abstract part def HeatingElement;
    part def ResistanceCoil :> HeatingElement {
        attribute resistance : Real default = 12.0;
    }
    part def PowerWire :> HeatingElement {
        attribute gauge : Real default = 14.0;
    }
    part def HeatingAssembly :> HeatingSystem {
        part coil : ResistanceCoil;
        part wire : PowerWire;
    }
    part heatingEvidence {
        assert satisfy heating by efficient;
        assert satisfy heating by weak;
    }
    part def BreadLoader { part bread : Start; }
    part def BreadEjector { part bread : Finish; }
    state Cycle {
        entry; then idle;
        state idle;
        state heating;
        state ready;
        state cancelled;
        transition first idle accept Start then heating;
        transition first heating accept Finish then ready;
        transition first heating accept Cancel then cancelled;
    }

    part def BreadHandling {
        part loader : BreadLoader;
        part ejector : BreadEjector;
        flow loader.bread to ejector.bread;
    }
}
"""

conn = opensysml.connect(version="v0.9.0")
model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"
print(f"Model ok: {model.ok}")


In [ ]:
# A bad constraint expression produces no failure verdicts (empty list or error).
bad_source = """
package P {
    private import ScalarValues::*;
    part def T { attribute x : Real default = 5.0; }
    requirement def R {
        subject t : T;
        require constraint { t.missingAttr <= 10.0 }
    }
}
"""
bad = conn.load_from_content(bad_source, strict=False)
assert not bad.ok, "Expected failure for undeclared attribute"
print(f"Negative control ok: bad.ok={bad.ok}")


In [ ]:
from toaster.evidence import ReviewRecord, hash_content, validate_record

# Locate the slow variant's failing verdict
verdicts = model.verify_satisfaction()
slow_verdict = next(
    (v for v in verdicts if "slow" in (v.element or "") and not v.holds),
    None,
)
assert slow_verdict is not None, f"Expected a failing slow verdict; got: {verdicts}"
print(f"Violation witness: {slow_verdict.element!r} holds={slow_verdict.holds}")

# Record the violation as a worked-example ReviewRecord (AS-C08)
violation = ReviewRecord(
    identifier="AS-C08",
    kind="asserted_solution",
    claim=(
        "The slow variant (cycleTime=200) violates TimelyToast (cycleTime ≤ 180): "
        "verify_satisfaction() returns holds=False for the slow candidate."
    ),
    model_ref="ToasterDemo::slow",
    content_hash=hash_content(source),
    scope="ToasterDemo",
    criteria="verify_satisfaction() returns holds=False for the slow candidate",
    premises=[],
    assumption_refs=["AS-C03"],
    evidence_refs=[slow_verdict.element or "satisfy timely by slow"],
    rationale=(
        "The Verdict from verify_satisfaction() is direct computational evidence. "
        "The model evaluates toaster.cycleTime <= 180.0 against slow.cycleTime=200.0, "
        "which is False. The violation is bounded to the defined cycleTime attribute; "
        "real toasters have variable cycle times depending on load and ambient temperature."
    ),
    counterevidence=(
        "The slow variant is a synthetic stress case, not a production design. "
        "A real toaster with cycleTime=200 might still satisfy a user if the toast "
        "is acceptable quality — the requirement captures one dimension of acceptability."
    ),
    residual_uncertainties=(
        "cycleTime is a fixed attribute; the model does not capture variation within "
        "a single toast cycle. Thermal modelling would be needed to assess that."
    ),
    disposition="pending",
    dependency_freshness="current",
    engineering_conclusion="refuted",
    record_kind="worked_example",
)

errors = validate_record(violation)
assert errors == [], f"Validation errors: {errors}"
print(f"Violation record valid: identifier={violation.identifier!r}")
print(f"engineering_conclusion={violation.engineering_conclusion!r}")
conn.close()


The slow variant's 200-second cycle time failing the TimelyToast constraint (A-F) is confirmed by the holds=False Verdict from `verify_satisfaction()` (O-S); the ReviewRecord captures this as a simulation-backed engineering judgment with a non-empty `counterevidence` field (E).


Try the chapter exercise in `exercises/ch08/exercise.ipynb`: produce a violation witness for the weak Heater variant against the HeatingReq requirement using `verify_satisfaction()`.
